In [1]:
import numpy as np
from StatTools.generators.ndfnoise_generator import ndfnoise
from StatTools.generators.multi_scale_fractional_generator import MultiScaleFractionalGenerator
import cv2 as cv
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd

def gen_traj(frame_num: int, ants_num: int, frame_shape: tuple, margin:int = 10,
             hurst_move: float = 0.5, hurst_species: float = 0.5, 
             start_point = None):
    

    dx = ndfnoise(shape=(frame_num, ants_num), hurst=[hurst_move, hurst_species], normalize=True, dtype=np.float32)
    dy = ndfnoise(shape=(frame_num, ants_num), hurst=[hurst_move, hurst_species], normalize=True, dtype=np.float32)

    x = dx.round().cumsum(axis=0).astype(np.int32)
    y = dy.round().cumsum(axis=0).astype(np.int32)

    if start_point is None:
        start_x = np.random.randint(margin, frame_shape[1] - margin, (ants_num,))
        start_y = np.random.randint(margin, frame_shape[0] - margin, (ants_num,))
        start_point = np.stack([start_x, start_y], axis=1)

    trajectories = np.stack([x, y], axis=2)
    trajectories = trajectories + start_point

    return trajectories

def gen_traj_corr(frame_num: int, ants_num: int, frame_shape: tuple, margin:int = 10,
                    h_list: list=None, crossover_points:list=None,
                    start_point = None, correlation_matrix: np.ndarray = None):
    
    if correlation_matrix is None:
        _r = np.random.rand(ants_num, ants_num)
        cov = _r @ _r.T
        var = cov.diagonal()
        d = np.sqrt(var)
        correlation_matrix = cov / np.outer(d,d)
        np.fill_diagonal(correlation_matrix, 1.0)
        
    generator = MultiScaleFractionalGenerator(h_list=h_list, crossover_points=crossover_points)
    dx = generator.generate(frame_num, ants_num, correlation_matrix=correlation_matrix).T
    dy = generator.generate(frame_num, ants_num, correlation_matrix=correlation_matrix).T

    x = dx.round().cumsum(axis=0).astype(np.int32)
    y = dy.round().cumsum(axis=0).astype(np.int32)

    if start_point is None:
        start_x = np.random.randint(margin, frame_shape[1] - margin, (ants_num,))
        start_y = np.random.randint(margin, frame_shape[0] - margin, (ants_num,))
        start_point = np.stack([start_x, start_y], axis=1)

    trajectories = np.stack([x, y], axis=2)
    trajectories = trajectories + start_point

    return trajectories

def draw_traj(trajectories, frame_shape:tuple, thickness:int):
    ants_num = trajectories.shape[1]
    bgs = [np.zeros(shape=frame_shape, dtype=np.uint8) for _ in range(ants_num)]
    trajs = []
    for ant in range(ants_num):
        trajs.append(np.array([np.array(traj) for traj in trajectories[:, ant]]))
    for ant, bg in zip(trajs, bgs):
        cv.polylines(bg, [ant], isClosed=False, color=(255), thickness=thickness)
    
    return bgs

In [2]:
trajectories_msfg = gen_traj_corr(frame_num=4000, ants_num=50, frame_shape=(500,500), h_list=[0.8,1.2], crossover_points=[1000])

In [3]:
trajectories_ndfnoise = gen_traj(frame_num=4000, ants_num=50, frame_shape=(500,500), hurst_move=1.0)

/home/akhiyarov/.local/lib/python3.12/site-packages/StatTools/generators/ndfnoise_generator.py:51: RuntimeWarning: divide by zero encountered in power
  S *= np.abs(f.reshape(reshape)) ** (-(hurst[i] - 0.5))
